# 03-building-blocks.py

03-building-blocks.py — Modulo 3: Building Blocks: como reutilizar lo existente (60 min)

In [0]:
%run ./_resource/00-setup

# Modulo 3 — Building Blocks: como reutilizar lo existente
**60 minutos** &nbsp;|&nbsp; Catalogo de componentes · Esquemas estandarizados · Clases Python · Asset Bundles · %run

En este modulo inspeccionamos los building blocks de Delfos, extendemos uno sin romper
contratos existentes, demostramos que los Delta Constraints protegen la integridad del
Data Product, y empaquetamos el pipeline como un Asset Bundle versionable en Git.

## Que es un Building Block en Delfos?

Un **building block** es un componente reutilizable que cualquier equipo puede usar sin
entender sus detalles internos. En Delfos hay tres tipos:

| Tipo | Ejemplos | Donde vive |
|---|---|---|
| **Esquemas** | Schema de transacciones, schema de alertas | Unity Catalog — `DESCRIBE TABLE` |
| **Transformaciones** | Limpieza de datos, deteccion de anomalias | `_resource/00-setup.py` via `%run` |
| **Pipelines** | Bronze→Silver, Silver→Gold | Asset Bundle en Git |

Principio: **no copiar, extender** — agrega un parametro al building block existente
en lugar de duplicar codigo.

### 3.1 — Inspeccionar el codigo fuente de un building block

Los building blocks son funciones Python definidas en `_resource/00-setup.py` e importadas
con `%run`. `inspect.getsource` permite ver el contrato antes de extenderlo.

### 3.1a — Ver el codigo fuente de aplicar_reglas_calidad

In [0]:
import inspect

print("=" * 65)
print("  Building Block: aplicar_reglas_calidad()")
print("  Propietario: equipo-datos | Tipo: Transformacion")
print("=" * 65)
print(inspect.getsource(aplicar_reglas_calidad))

### 3.1b — Catalogo de Building Blocks en Unity Catalog

In [0]:
# Inventario completo de Data Products del catalogo del participante.
# Cualquier ingeniero puede descubrir lo disponible sin documentacion externa.
spark.sql(f"""
    SELECT
        table_schema   AS dominio,
        table_name     AS componente,
        table_type     AS tipo,
        comment        AS descripcion
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema != 'information_schema'
    ORDER BY dominio, componente
""").display()

### 3.2 — Extender un building block sin romper contratos existentes

El equipo de riesgo necesita `zscore_por_usuario` con fallback al **canal** para usuarios
sin historial propio. Regla de Delfos: **misma firma de entrada y salida que v1** —
cualquier pipeline existente puede adoptar v2 cambiando solo el nombre de la funcion.

### 3.2a — Definir zscore_por_usuario_v2 con fallback al canal

In [0]:
def zscore_por_usuario_v2(df: DataFrame,
                           umbral: float = 3.0,
                           fallback_canal: bool = True) -> DataFrame:
    """
    Building block extendido: z-score con fallback al canal para usuarios sin historial.
    Compatible con v1 — mismos parametros de entrada y salida garantizados.
    """
    v_usuario = Window.partitionBy("user_id")
    v_canal   = Window.partitionBy("canal")

    return (df
        .withColumn("_prom_usr",   F.avg("monto").over(v_usuario))
        .withColumn("_std_usr",    F.stddev("monto").over(v_usuario))
        .withColumn("_prom_canal", F.avg("monto").over(v_canal))
        .withColumn("_std_canal",  F.stddev("monto").over(v_canal))
        .withColumn("_tiene_historial",
            F.col("_std_usr").isNotNull() & (F.col("_std_usr") > 0))
        .withColumn("_prom_final",
            F.when(F.col("_tiene_historial"), F.col("_prom_usr"))
             .when(F.lit(fallback_canal),     F.col("_prom_canal"))
             .otherwise(F.col("_prom_usr")))
        .withColumn("_std_final",
            F.when(F.col("_tiene_historial"), F.col("_std_usr"))
             .when(F.lit(fallback_canal),     F.col("_std_canal"))
             .otherwise(F.col("_std_usr")))
        .withColumn("alerta_zscore",
            F.when(F.col("_std_final").isNotNull() & (F.col("_std_final") > 0),
                   (F.col("monto") - F.col("_prom_final")) / F.col("_std_final") > umbral
            ).otherwise(F.lit(False)))
        .drop("_prom_usr","_std_usr","_prom_canal","_std_canal",
              "_tiene_historial","_prom_final","_std_final")
    )

print("zscore_por_usuario_v2 definida — firma identica a v1: columna 'alerta_zscore' (boolean).")

### 3.2b — Comparar alertas v1 vs v2: cuanto captura el fallback

In [0]:
df_test = spark.table(f"{CATALOG}.{SCH_PAGOS}.transacciones").filter("capa='silver'").limit(5000)
n_v1 = zscore_por_usuario(df_test).filter("alerta_zscore").count()
n_v2 = zscore_por_usuario_v2(df_test, fallback_canal=True).filter("alerta_zscore").count()

print(f"Muestra: {df_test.count():,} transacciones Silver")
print(f"  v1 (historial del usuario) : {n_v1:>6,} alertas")
print(f"  v2 (fallback al canal)     : {n_v2:>6,} alertas")
print(f"  Alertas adicionales        : {n_v2 - n_v1:>6,}  (usuarios nuevos sin historial propio)")

# Verificar compatibilidad de firma: mismo tipo de salida que v1
col_v1 = dict(zscore_por_usuario(df_test).dtypes)["alerta_zscore"]
col_v2 = dict(zscore_por_usuario_v2(df_test).dtypes)["alerta_zscore"]
print(f"\nCompatibilidad de firma: {'OK — mismo tipo de salida' if col_v1 == col_v2 else 'ERROR'}")

### 3.3 — Delta Constraints: el contrato tecnico protege la integridad

Los constraints de Delta son la especificacion tecnica del contrato del Data Product.
Si un dato viola el constraint, Delta **rechaza toda la escritura** con un error explicito.
La siguiente celda intenta insertar datos invalidos intencionalmente — debe fallar.

### 3.3a — Agregar Delta Constraints al Data Product

In [0]:
# DROP + ADD para hacerlo idempotente (ADD CONSTRAINT no soporta IF NOT EXISTS).
# Usa las variables CATALOG y SCH_PAGOS del setup — nunca hardcodear el catalogo.
_t = f"{CATALOG}.{SCH_PAGOS}.transacciones"
for _constraint, _check in [
    ("monto_positivo", "monto > 0"),
    ("canal_valido",   "canal IN ('app','qr','corresponsal','api')"),
    ("ts_no_futuro",   "ts <= current_timestamp()"),
]:
    spark.sql(f"ALTER TABLE {_t} DROP CONSTRAINT IF EXISTS {_constraint}")
    spark.sql(f"ALTER TABLE {_t} ADD CONSTRAINT {_constraint} CHECK ({_check})")
    print(f"  Constraint '{_constraint}': OK")

print()
spark.sql(f"DESCRIBE DETAIL {_t}").display()

### 3.3b — Demostrar que el constraint rechaza datos invalidos

In [0]:
# Este bloque DEBE fallar — demuestra que el contrato tecnico funciona.
import pandas as _pd_test
from datetime import datetime as _dt

for _caso, _row in [
    ("monto negativo",   {"monto": -500.0,  "canal": "app",      "transaction_id": "TEST-001"}),
    ("canal invalido",   {"monto": 50000.0, "canal": "whatsapp", "transaction_id": "TEST-002"}),
]:
    _datos = _pd_test.DataFrame([{
        **_row,
        "user_id": "u_9999", "ciudad": "Bogota", "dispositivo": "Android",
        "ts": "2020-01-01T00:00:00", "capa": "silver",
        "es_fraude_real": False, "_procesado_en": _dt.now(),
    }])
    try:
        spark.createDataFrame(_datos).write.format("delta").mode("append") \
             .saveAsTable(f"{CATALOG}.{SCH_PAGOS}.transacciones")
        print(f"  {_caso:<20} ERROR: el dato invalido se escribio")
    except Exception as e:
        print(f"  {_caso:<20} OK — Delta rechazo la escritura ({type(e).__name__})")

### 3.4 — Importar building blocks: %run vs dbutils.notebook.run

| Mecanismo | Contexto | Parametros | Cuando usarlo |
|---|---|---|---|
| `%run` | Mismo notebook | No permite | Cargar funciones y variables compartidas |
| `dbutils.notebook.run` | Proceso separado | Si permite | Pipelines completos con timeout y retorno de metricas |

Despues del `%run ./_resource/00-setup` al inicio, todas las funciones y variables
de setup quedan disponibles como si estuvieran definidas localmente.

### 3.4a — Verificar building blocks disponibles en el contexto actual

In [0]:
import inspect

bloques_disponibles = {
    "aplicar_reglas_calidad" : aplicar_reglas_calidad,
    "zscore_por_usuario"     : zscore_por_usuario,
    "frecuencia_ventana"     : frecuencia_ventana,
}

print("Building blocks importados por %run ./_resource/00-setup:")
print()
for nombre, fn in bloques_disponibles.items():
    sig    = inspect.signature(fn)
    n_args = len(sig.parameters)
    print(f"  {nombre:<30} {n_args} parametro(s): {list(sig.parameters.keys())}")

print()
print("Variables de entorno disponibles:")
print(f"  CATALOG      = {CATALOG}")
print(f"  PATH_BRONZE  = {PATH_BRONZE}")
print(f"  PATH_SILVER  = {PATH_SILVER}")
print(f"  PATH_CKPT    = {PATH_CKPT}")

### 3.5 — Asset Bundle: el pipeline como codigo versionable en Git

Un **Databricks Asset Bundle** empaqueta notebooks, jobs y configuracion en un artefacto
versionable en Git. El `databricks.yml` define todo lo necesario para desplegar el pipeline
en cualquier entorno con un solo comando desde la CLI.

- Pipeline como **codigo**: revisable en PR, auditable en git log
- Target `dev` prefija `dev_` al job — sin interferir con produccion
- Variables (`umbral_z`, `max_tx`) parametrizables sin tocar los notebooks

### 3.5a — Definir y guardar el databricks.yml del Asset Bundle

In [0]:
YAML_BUNDLE = f"""\
# databricks.yml — Asset Bundle de Delfos M1
bundle:
  name: delfos-m1-pipeline

variables:
  catalog:
    default: {CATALOG}
    description: Catalogo Unity Catalog
  umbral_z:
    default: "3.0"
    description: Umbral z-score
  max_tx:
    default: "5"
    description: Max transacciones en 10 min

targets:
  dev:
    mode: development
    workspace:
      host: ${{workspace_host}}
  prod:
    mode: production
    workspace:
      host: ${{workspace_host}}

resources:
  jobs:
    delfos_pipeline_pagos:
      name: "Delfos - Pipeline Pagos (Bronze->Silver->Gold)"
      schedule:
        quartz_cron_expression: "0 0 11 * * ?"
        timezone_id: "America/Bogota"
      tasks:
        - task_key: bronze_to_silver
          notebook_task:
            notebook_path: ./02-arquitectura-aws-databricks-airflow
            source: WORKSPACE
            base_parameters:
              catalog: ${{var.catalog}}
              reset: "No"
          new_cluster:
            spark_version: 15.4.x-scala2.12
            num_workers: 2
            data_security_mode: SINGLE_USER
        - task_key: silver_to_gold
          depends_on:
            - task_key: bronze_to_silver
          notebook_task:
            notebook_path: ./02-arquitectura-aws-databricks-airflow
            source: WORKSPACE
            base_parameters:
              catalog: ${{var.catalog}}
              umbral_z: ${{var.umbral_z}}
              max_tx: ${{var.max_tx}}
          new_cluster:
            spark_version: 15.4.x-scala2.12
            num_workers: 2
            data_security_mode: SINGLE_USER
"""

import os
_nb_path      = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_project_root = "/".join(_nb_path.split("/")[:-1])
_bundle_path  = f"/Workspace{_project_root}/databricks.yml"

os.makedirs(os.path.dirname(_bundle_path), exist_ok=True)
with open(_bundle_path, "w") as f:
    f.write(YAML_BUNDLE)

print(f"databricks.yml guardado en: {_bundle_path}")

### 3.5b — Ciclo de vida del Asset Bundle

In [0]:
print("Ciclo de vida del Asset Bundle en Delfos:")
print()
print("  1. Validar:  $ databricks bundle validate")
print("  2. Dev:      $ databricks bundle deploy --target dev")
print("  3. Ejecutar: $ databricks bundle run delfos_pipeline_pagos --target dev")
print("  4. Params:   $ databricks bundle run delfos_pipeline_pagos \\")
print('                    --python-params \'{"umbral_z":"3.5","max_tx":"4"}\'')
print("  5. Destruir: $ databricks bundle destroy --target dev")
print("  6. Prod:     $ databricks bundle deploy --target prod")

---

## Resumen del modulo

| Que hiciste | Herramienta | Por que importa |
|---|---|---|
| Inspeccionar codigo fuente de building blocks | `inspect.getsource()` | Entender el contrato antes de extender |
| Consultar inventario de Data Products | `information_schema.tables` | Descubrimiento self-serve del catalogo |
| Extender zscore sin romper contratos | nueva funcion v2 con misma firma | Principio Open/Closed en Data Products |
| Agregar Delta Constraints | `ADD CONSTRAINT CHECK` | Contrato tecnico inmutable en la tabla |
| Probar constraints con datos invalidos | monto negativo + canal invalido | Demostrar que la proteccion funciona |
| Verificar funciones disponibles via %run | `inspect.signature` | Entender el mecanismo de importacion |
| Generar y desplegar Asset Bundle | `databricks.yml` + CLI | Pipeline como codigo versionable en Git |

---

## Reflexion

El equipo de clientes quiere detectar usuarios inactivos reactivados. Extenderias
`zscore_por_usuario_v2` o crearias un building block nuevo en `nequi_prod.clientes`?
Que principio de Data Mesh aplica en esa decision?

**Referencia:** [Databricks Asset Bundles](https://docs.databricks.com/en/dev-tools/bundles/index.html)